In [1]:
# Importe

import re
import pandas as pd
import numpy as np


### RIS

In [2]:
# RIS lesen

RIS_PATH = "../../data/metadata/raw/gbv-ris.ris"

def read_ris_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content

ris_file = read_ris_file(RIS_PATH)

In [3]:
# RIS zu DataFrame

def ris_to_df(ris):
    rows = []
    eintrag = {}
    key_counts = {}
    
    for line in ris.splitlines():
        line = line.strip()
        if line == "ER  -":
            rows.append(eintrag)
            eintrag = {}
            key_counts = {}
        elif "  - " in line:
            key, value = line.split("  - ", 1)
            key = key.strip()
            value = value.strip()
            if key in eintrag:
                i = 2
                while f"{key}_{i}" in eintrag:
                    i += 1
                eintrag[f"{key}_{i}"] = value
            else:
                eintrag[key] = value
    
    return pd.DataFrame(rows)

ris_df = ris_to_df(ris_file)


# RIS-Datenbereinigung

mask = ris_df['A2'].notna()
ris_df.loc[mask, 'A1'] = ris_df.loc[mask, 'A2']
ris_df = ris_df.drop(columns=["PB", 
                              "CY",
                              "A2", 
                              "U1", 
                              "TY", 
                              "H2", "H2_2", 
                              "S1", "S2", 
                              "L3",  
                              "KW", "KW_2",
                              "T2", "T2_2", "T2_3", "T2_4", "T2_5", "T2_6", "T2_7", "T3", 
                              "N1", "N1_2", "N1_3"])

# PY (Erscheinungsjahr) bereinigen: einzelne Ausreisser (z.B. 2001 = Digitalisierungsjahr
# des DFG-Projekts, faelschlich als PY erfasst -- betrifft nur 11 von ~5986 Zeilen)
# ausserhalb des plausiblen Zeitraums der Schriftenreihen (1700-1900) verwerfen
py_numeric = pd.to_numeric(ris_df['PY'], errors='coerce')
ris_df.loc[(py_numeric < 1700) | (py_numeric > 1900), 'PY'] = None

def normalize_caps(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return value
    
    def fix_word(word):
        if word.isupper():
            return word.capitalize()
        return word

    return ' '.join(fix_word(w) for w in str(value).split(' '))

ris_df['A1'] = ris_df['A1'].apply(normalize_caps)

In [4]:
# Vollanzeige lesen und in Liste splitten

VA_PATH = "../../data/metadata/raw/gbv-vollanzeige.txt"

def read_va_file(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
    return content

va_file = read_va_file(VA_PATH)
va_file = re.split(r'(?=Aufsatz / Teil:)', va_file.strip())
va_file.pop(0) 

''

### VA

In [5]:
# VA zu Dataframe

labels = [
    'Aufsatz / Teil:',
    'VerfasserIn:',
    'Veröffentlichungsangabe:',
    'Umfang:',
    'Anmerkung:',
    'Bibliogr. Zusammenhang:',
    'Gesamttitel:',
    'Volltext:',
    'Lokale Schlagwörter:',
    'Signatur:',
    'Schlagwörter:',
    'Standort:',
    'Schriftenreihe:'   
]

pattern = '(' + '|'.join(re.escape(label) for label in labels) + ')'

va_liste_neu = []
for eintrag in va_file:
    eintrag = re.sub(r'\s+', ' ', eintrag).strip()
    teile = re.split(pattern, eintrag)
    
    row = {}
    for i in range(1, len(teile), 2):
        key = teile[i].strip().rstrip(':')
        value = teile[i+1].strip() if i+1 < len(teile) else ''
        if not value:
            continue
        if key in row:
            i_count = 2
            while f"{key}_{i_count}" in row:
                i_count += 1
            row[f"{key}_{i_count}"] = value
        else:
            row[key] = value
    
    va_liste_neu.append(row)

va_df = pd.DataFrame(va_liste_neu)

In [6]:
va_df

,Aufsatz / Teil,VerfasserIn,Veröffentlichungsangabe,Veröffentlichungsangabe_2,Gesamttitel,Volltext,Signatur,Anmerkung,Schlagwörter,Bibliogr. Zusammenhang,...,Bibliogr. Zusammenhang_2,Schlagwörter_2,Gesamttitel_3,Volltext_3,Standort_2,Signatur_3,Anmerkung_2,VerfasserIn_2,Schriftenreihe,Anmerkung_3
0,Die elektromagnetische Drehung der Polarisatio...,"Kundt, August *1839-1894*",2001 (Original: 1884),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 350,Vgl.: Zweite Mittheilung unter dem Titel: Über...,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Über die Mittheilung des Tones longitudinal sc...,"KUNDT, August",2001 (Original: 1865),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,NaN,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Über Erzeugung stehender Schwingungen und Klan...,"KUNDT, August",2001 (Original: 1868),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,NaN,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Epigraphische Reiseberichte aus Spanien und Po...,"Hübner, Emil *1834-1901*",2001,Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Vgl. 1860. MB S. 231-241; 324-332; 421-450; 59...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Über einige Bestandtheile der peripheren markh...,"Joseph, Max *1860-1932*",2001 (Original: 1888),Berlin : Berlin-Brandenburgische Akademie der ...,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 350,NaN,NaN,In: Sitzungsberichte der Königlich Preußischen...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8298,Über Potenzreihen,"KRONECKER, Leopold",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8299,Über die Ossification der Geweihe,"LIEBERKÜHN, Nathanael",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8300,Über eine Untersuchung des sogenannten capitol...,"JORDAN, Heinrich",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8301,"Über eine neue, mit Halieutaea verwandte Fisch...","PETERS, Wilhelm",Berlin : Berlin-Brandenburgische Akademie der ...,NaN,Digitalisierte Akademieschriften,<a href='https://digilib.bbaw.de/digitallibrar...,Z 349,Monatsberichte der Königlich Preußischen Akade...,Akademiemitglied_v,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,mit 1 Tafel,NaN,NaN,NaN


In [7]:
# VA-Daten löschen

# Spalten löschen
mask = va_df['Anmerkung'].isna() & va_df['Umfang'].notna()
va_df.loc[mask, 'Anmerkung'] = va_df.loc[mask, 'Umfang']

va_df = va_df.drop(columns=["Volltext_2", "Volltext_3", 
                            "Schriftenreihe",
                            "Aufsatz / Teil", 
                            "Signatur", "Signatur_2", "Signatur_3", 
                            "Standort", "Standort_2", 
                            "Gesamttitel", "Gesamttitel_2", "Gesamttitel_3", 
                            "Schlagwörter", "Schlagwörter_2", "Lokale Schlagwörter",
                            "Veröffentlichungsangabe", "Veröffentlichungsangabe_2", "Veröffentlichungsangabe_3", "Veröffentlichungsangabe_4",
                            "VerfasserIn", "VerfasserIn_2",
                            "Bibliogr. Zusammenhang_2",
                            "Anmerkung_3",
                            "Umfang", "Umfang_2"
                            ])



In [8]:
va_df

,Volltext,Anmerkung,Bibliogr. Zusammenhang,Anmerkung_2
0,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl.: Zweite Mittheilung unter dem Titel: Über...,NaN,NaN
1,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,NaN
2,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,NaN
3,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl. 1860. MB S. 231-241; 324-332; 421-450; 59...,NaN,NaN
4,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,In: Sitzungsberichte der Königlich Preußischen...,NaN
...,...,...,...,...
8298,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN
8299,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN
8300,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,NaN
8301,<a href='https://digilib.bbaw.de/digitallibrar...,Monatsberichte der Königlich Preußischen Akade...,NaN,mit 1 Tafel


In [9]:
# Anmerkungen und Bibliographischen Zusammenhang mergen

# Bibliographischen Zusammenhang aufräumen
suchstrings_bib_zus = "Erscheint auch als|Sonderdruck|Druckausg|Digitalisierte Akademieschriften"
maske1 = va_df["Bibliogr. Zusammenhang"].str.contains(suchstrings_bib_zus, na=False)
leer1 = maske1 & va_df["Anmerkung_2"].isna()
gefuellt1 = maske1 & va_df["Anmerkung_2"].notna()
va_df.loc[leer1, "Anmerkung_2"] = va_df.loc[leer1, "Anmerkung"]
va_df.loc[gefuellt1, "Anmerkung_2"] = va_df.loc[gefuellt1, "Anmerkung_2"] + "; " + va_df.loc[gefuellt1, "Anmerkung"]
va_df.loc[maske1, "Bibliogr. Zusammenhang"] = np.nan
va_df["Bibliogr. Zusammenhang"] = va_df["Bibliogr. Zusammenhang"].str.split(";").str[-1]
va_df.loc[va_df["Bibliogr. Zusammenhang"].str.contains("In:", na=False), "Bibliogr. Zusammenhang"] = np.nan

# Anmerkungen aufräumen
suchstrings_anmerkung = "Tafel|Tabelle|Blätter|Rezension|Réponse|Neue Versuche|Nachträge|Nachtrag|Vgl.|Vergl.|Fortsetzung|Auszug|Antwort|Zweiter Theil|Zweiter Bericht|Zweite Mittheilung"
maske2 = va_df["Anmerkung"].str.contains(suchstrings_anmerkung, na=False)
leer2 = maske2 & va_df["Anmerkung_2"].isna()
gefuellt2 = maske2 & va_df["Anmerkung_2"].notna()
va_df.loc[leer2, "Anmerkung_2"] = va_df.loc[leer2, "Anmerkung"]
va_df.loc[gefuellt2, "Anmerkung_2"] = va_df.loc[gefuellt2, "Anmerkung_2"] + "; " + va_df.loc[gefuellt2, "Anmerkung"]
va_df.loc[maske2, "Anmerkung"] = np.nan
va_df["Anmerkung"] = va_df["Anmerkung"].str.split("Übers.").str[0]
va_df["Anmerkung"] = va_df["Anmerkung"].str.split("Veröffentlichungsangabe").str[0]
va_df["Anmerkung"] = va_df["Anmerkung"].str.split(";").str[-1]

# Bibliographischer Zusammenhang und Anmerkung mergen
maske = va_df["Bibliogr. Zusammenhang"].notna()
va_df.loc[maske, "Anmerkung"] = va_df.loc[maske, "Bibliogr. Zusammenhang"]
va_df = va_df.drop(columns=["Bibliogr. Zusammenhang"])
va_df["Seiten"] = va_df["Anmerkung"]
va_df = va_df.drop(columns=["Anmerkung"])
va_df["Anmerkung"] = va_df["Anmerkung_2"]
va_df = va_df.drop(columns=["Anmerkung_2"])
va_df["Anmerkung"] = va_df["Anmerkung"].str.strip("\"'„“‚’")

In [10]:
# Seiten bearbeiten

# Fortsetzung aus Seiten extrahieren
maske = va_df["Seiten"].str.contains(" und ", na=False)
va_df.loc[maske, "Fortsetzung"] = "ebenda. " + va_df.loc[maske, "Seiten"].str.split(" und ").str[1]
va_df.loc[maske, "Seiten"] = va_df.loc[maske, "Seiten"].str.split(" und ").str[0]

# Klassen aus Seiten extrahieren (und droppen)
va_df["Klasse"] = va_df["Seiten"].str.extract(r"^([^,]*,)")
va_df["Seiten"] = va_df["Seiten"].str.replace(r"^[^,]*,", "", regex=True)
va_df["Seiten"] = va_df["Seiten"].str.replace(r"\s+", "", regex=True)
va_df["Klasse"] = va_df["Klasse"].str.strip().str.rstrip(",")
va_df = va_df.drop(columns=["Klasse"])

# SP und EP extrahieren
def berechne_sp_ep(df, spalte, sp_name, ep_name):
    # Klammern entfernen
    df[spalte] = df[spalte].str.replace(r"[()]", "", regex=True)

    # Maske: Zeilen, in denen die Spalte mit "S" beginnt
    maske_s = df[spalte].str.startswith("S", na=False)
    werte_s = df.loc[maske_s, spalte].str.findall(r"\d+|[IVXLCDM]+"
)

    # Fall 1: nur ein Wert -> SP = EP
    einzelwert = werte_s[werte_s.str.len() == 1]
    df.loc[einzelwert.index, sp_name] = einzelwert.str[0]
    df.loc[einzelwert.index, ep_name] = einzelwert.str[0]

    # Fall 2: zwei Werte -> SP = erster, EP = zweiter
    zweiwerte = werte_s[werte_s.str.len() >= 2]
    df.loc[zweiwerte.index, sp_name] = zweiwerte.str[0]
    df.loc[zweiwerte.index, ep_name] = zweiwerte.str[1]

    # Maske: Zeilen, die NICHT mit "S" beginnen
    maske_kein_s = ~df[spalte].str.startswith("S", na=False)

    # Fall 3: Werte mit "-" getrennt -> SP = erster, EP = zweiter
    maske_bindestrich = maske_kein_s & df[spalte].str.contains("-", na=False)
    werte_bindestrich = df.loc[maske_bindestrich, spalte].str.findall(r"\d+|[IVXLCDM]+"
)
    df.loc[werte_bindestrich.index, sp_name] = werte_bindestrich.str[0]
    df.loc[werte_bindestrich.index, ep_name] = werte_bindestrich.str[1]

    # Fall 4: "S" kommt nach dem Wert -> SP = 1, EP = Wert
    maske_s_danach = maske_kein_s & ~df[spalte].str.contains("-", na=False) & df[spalte].str.contains("S", na=False)
    werte_s_danach = df.loc[maske_s_danach, spalte].str.findall(r"\d+|[IVXLCDM]+"
)
    df.loc[werte_s_danach.index, sp_name] = "1"
    df.loc[werte_s_danach.index, ep_name] = werte_s_danach.str[0]

    # Fall 5: kein "-", kein "S", ein Wert -> SP = 1, EP = Wert
    maske_nur_wert = maske_kein_s & ~df[spalte].str.contains("-", na=False) & ~df[spalte].str.contains("S", na=False)
    werte_nur_wert = df.loc[maske_nur_wert, spalte].str.findall(r"\d+|[IVXLCDM]+"
)
    df.loc[werte_nur_wert.index, sp_name] = "1"
    df.loc[werte_nur_wert.index, ep_name] = werte_nur_wert.str[0]

    # SP > EP -> erste Ziffer von SP entfernen (nur bei arabischen Zahlen wirksam,
    # da pd.to_numeric römische Ziffern zu NaN macht und der Vergleich dann False ergibt)
    sp_num = pd.to_numeric(df[sp_name], errors="coerce")
    ep_num = pd.to_numeric(df[ep_name], errors="coerce")
    maske_sp_groesser = sp_num > ep_num
    df.loc[maske_sp_groesser, sp_name] = df.loc[maske_sp_groesser, sp_name].str[1:]

    # Fall: Spalte leer/NaN -> SP, EP explizit leeren (überschreibt alles Vorherige)
    maske_leer = df[spalte].isna() | (df[spalte].str.strip() == "")
    df.loc[maske_leer, sp_name] = None
    df.loc[maske_leer, ep_name] = None

# Aufruf für beide Spalten
berechne_sp_ep(va_df, "Seiten", "SP", "EP")
va_df = va_df.drop(columns=["Seiten"])

va_df 

,Volltext,Anmerkung,Fortsetzung,SP,EP
0,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl.: Zweite Mittheilung unter dem Titel: Über...,NaN,None,None
1,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,None,None
2,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,None,None
3,<a href='https://digilib.bbaw.de/digitallibrar...,Vgl. 1860. MB S. 231-241; 324-332; 421-450; 59...,NaN,None,None
4,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,1321,1330
...,...,...,...,...,...
8298,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,53,58
8299,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,264,267
8300,<a href='https://digilib.bbaw.de/digitallibrar...,NaN,NaN,526,548
8301,<a href='https://digilib.bbaw.de/digitallibrar...,mit 1 Tafel,NaN,736,742


In [11]:
# Informationen aus Anmerkungen extrahieren

# Tafel-Angaben extrahieren
maske_mit = va_df["Anmerkung"].str.match(r"^(mit|und)", na=False)
pattern_mit_trenner = r"^((?:mit|und).*?)(?:(?<!\w)-(?!\w)|(?<!Fig)(?<!\d)\.)"
extrahiert = va_df.loc[maske_mit, "Anmerkung"].str.extract(pattern_mit_trenner)[0]
maske_kein_trenner = extrahiert.isna()
extrahiert.loc[maske_kein_trenner] = va_df.loc[maske_mit, "Anmerkung"][maske_kein_trenner]
va_df.loc[maske_mit, "Abbilder"] = extrahiert
va_df.loc[maske_mit, "Anmerkung"] = va_df.loc[maske_mit, "Anmerkung"].str.replace(pattern_mit_trenner, "", regex=True, n=1)
va_df.loc[maske_mit & maske_kein_trenner.reindex(va_df.index, fill_value=False), "Anmerkung"] = ""

# Übersetzungsinformationen extrahieren (bleibt erhalten, wird spaeter in Textbeziehungen aufgeloest)
pattern_ueb = r"(Übersetzung.*|Deutsch unter dem Titel.*|In französischer Übersetzung.*|Original unter dem Titel.*)$"
maske_ueb = va_df["Anmerkung"].str.contains(
    r"Übersetzung|Deutsch unter dem Titel|In französischer Übersetzung|Original unter dem Titel",
    na=False
)
va_df.loc[maske_ueb, "Übersetzung/Original"] = va_df.loc[maske_ueb, "Anmerkung"].str.extract(pattern_ueb)[0]
va_df.loc[maske_ueb, "Anmerkung"] = va_df.loc[maske_ueb, "Anmerkung"].str.replace(pattern_ueb, "", regex=True)

# Anworten extrahieren
pattern_ant = r"(Antwort von.*|Autwort von.*|Résponse de.*)$"
maske_ant = va_df["Anmerkung"].str.contains(
    r"Antwort von.*|Autwort von.*|Résponse de.*",
    na=False
)
va_df.loc[maske_ant, "Antwort"] = va_df.loc[maske_ant, "Anmerkung"].str.extract(pattern_ant)[0]
va_df.loc[maske_ant, "Anmerkung"] = va_df.loc[maske_ant, "Anmerkung"].str.replace(pattern_ant, "", regex=True)




In [12]:
# Reinigungen mit Informationsverlust
va_df["Anmerkung"] = va_df["Anmerkung"].str.replace(r"[\[\]()]", "", regex=True)
va_df["Anmerkung"] = va_df["Anmerkung"].str.strip()
va_df["Anmerkung"] = va_df["Anmerkung"].str.replace(r"^-|-$", "", regex=True)
va_df["Anmerkung"] = va_df["Anmerkung"].str.strip()
va_df["Anmerkung"] = va_df["Anmerkung"].replace("", np.nan)

va_df["Abbilder"] = va_df["Abbilder"].str.replace(r"(\[|;|Vgl|Second|die|und SB|\().*$", "", regex=True)
va_df["Abbilder"] = va_df["Abbilder"].str.strip().str.rstrip(".")

maske_loeschen = va_df["Anmerkung"].str.match(r"^(\d|In franzosisch|Abhandlungen|Abh.|=|Bezeichnet als|Bericht|Einige|Druckfehle|Monatsberichte|Uranios|S.|SB|Auch in Formey|Addition|Anhang zu dieser Abhandlung)", na=False, case=False)
va_df.loc[maske_loeschen, "Anmerkung"] = np.nan

### MERGE

In [13]:
# RIS und VA zusammenführen

# Link für Merge herstellen
def extract_link(text):
    if pd.isna(text):
        return text
    match = re.search(r"href=['\"](.*?)['\"]", text)
    if match:
        link = match.group(1)
        link = re.sub(r'\s+', '', link)  # entfernt alle Leerzeichen/Umbrüche im Link
        return link
    return None

va_df['Volltext'] = va_df['Volltext'].apply(extract_link)
merged_df = pd.merge(
    va_df, ris_df,
    left_on="Volltext", right_on="UR",
    how="outer", indicator=True
)

# Reinigung
merged_df = merged_df.drop(columns=["Volltext", "_merge"])
merged_df = merged_df.drop_duplicates(subset=["UR"], keep="first")
merged_df = merged_df[merged_df['ID'] != "53886186X"]

# Schriftenreihe und Band aus der UR extrahieren
pattern = r'Bibliothek\.tiff/([^/]+)/([^/]+)/tif/'
extracted = merged_df['UR'].str.extract(pattern)
merged_df['SR'] = extracted[0]
merged_df['BD'] = extracted[1]

In [14]:
# Seiten manuell nachtragen

merged_df.loc[merged_df["ID"] == "094932654", "SP"] = "1"
merged_df.loc[merged_df["ID"] == "094945292", "SP"] = "1"
merged_df.loc[merged_df["ID"] == "094945292", "EP"] = "176"
merged_df.loc[merged_df["ID"] == "094958130", "SP"] = "111"
merged_df.loc[merged_df["ID"] == "09496775X", "SP"] = "394"
merged_df.loc[merged_df["ID"] == "09496775X", "SP"] = "416"
merged_df.loc[merged_df["ID"] == "094917906", "SP"] = "187"
merged_df.loc[merged_df["ID"] == "094917906", "EP"] = "187"
merged_df.loc[merged_df["ID"] == "094955220", "SP"] = "0"
merged_df.loc[merged_df["ID"] == "094955220", "EP"] = "0"
merged_df.loc[merged_df["ID"] == "094969892", "SP"] = "0"
merged_df.loc[merged_df["ID"] == "094969892", "EP"] = "0"
merged_df.loc[merged_df["ID"] == "094969892", "SP"] = "I"
merged_df.loc[merged_df["ID"] == "094969892", "EP"] = "LXXX"
merged_df.loc[merged_df["ID"] == "094902305", "SP"] = "329"
merged_df.loc[merged_df["ID"] == "094902305", "EP"] = "359"
merged_df.loc[merged_df["ID"] == "094926573", "SP"] = "677"
merged_df.loc[merged_df["ID"] == "094926573", "EP"] = "706"
merged_df.loc[merged_df["ID"] == "094971730", "SP"] = "329"
merged_df.loc[merged_df["ID"] == "094971730", "EP"] = "362"
merged_df.loc[merged_df["ID"] == "094972605", "SP"] = "628"
merged_df.loc[merged_df["ID"] == "094972605", "EP"] = "636"
merged_df.loc[merged_df["ID"] == "094977550", "SP"] = "639"
merged_df.loc[merged_df["ID"] == "09495688X", "SP"] = "248"
merged_df.loc[merged_df["ID"] == "09495688X", "EP"] = "252"
merged_df.loc[merged_df["ID"] == "094969213", "SP"] = "234"
merged_df.loc[merged_df["ID"] == "094969213", "EP"] = "255"
merged_df.loc[merged_df["ID"] == "094936366", "SP"] = "62"
merged_df.loc[merged_df["ID"] == "094936366", "EP"] = "70"
merged_df.loc[merged_df["ID"] == "094897212", "SP"] = "747"
merged_df.loc[merged_df["ID"] == "094897212", "EP"] = "772"
merged_df.loc[merged_df["ID"] == "094904707", "SP"] = "82"
merged_df.loc[merged_df["ID"] == "094904707", "EP"] = "119"
merged_df.loc[merged_df["ID"] == "094966303", "SP"] = "123"
merged_df.loc[merged_df["ID"] == "094966303", "EP"] = "130"
merged_df.loc[merged_df["ID"] == "094896968", "SP"] = "629"
merged_df.loc[merged_df["ID"] == "094896968", "EP"] = "640"
merged_df.loc[merged_df["ID"] == "094925887", "SP"] = "759"
merged_df.loc[merged_df["ID"] == "094925887", "EP"] = "763"
merged_df.loc[merged_df["ID"] == "094907757", "SP"] = "100"
merged_df.loc[merged_df["ID"] == "094907757", "EP"] = "106"
merged_df.loc[merged_df["ID"] == "094969604", "SP"] = "87"
merged_df.loc[merged_df["ID"] == "094969604", "EP"] = "89"
merged_df.loc[merged_df["ID"] == "094976732", "SP"] = "761"
merged_df.loc[merged_df["ID"] == "094976732", "EP"] = "782"
merged_df.loc[merged_df["ID"] == "094903123", "SP"] = "781"
merged_df.loc[merged_df["ID"] == "094903123", "EP"] = "786"
merged_df.loc[merged_df["ID"] == "094925895", "SP"] = "1035"
merged_df.loc[merged_df["ID"] == "094925895", "EP"] = "1042"
merged_df.loc[merged_df["ID"] == "09492483X", "SP"] = "21"
merged_df.loc[merged_df["ID"] == "09492483X", "EP"] = "32"
merged_df.loc[merged_df["ID"] == "094900248", "SP"] = "515"
merged_df.loc[merged_df["ID"] == "094900248", "EP"] = "526"
merged_df.loc[merged_df["ID"] == "094897174", "SP"] = "659"
merged_df.loc[merged_df["ID"] == "094897174", "EP"] = "664"
merged_df.loc[merged_df["ID"] == "094911495", "SP"] = "665"
merged_df.loc[merged_df["ID"] == "094911495", "EP"] = "667"
merged_df.loc[merged_df["ID"] == "094932069", "SP"] = "782"
merged_df.loc[merged_df["ID"] == "094932069", "EP"] = "815"
merged_df.loc[merged_df["ID"] == "094919364", "SP"] = "231"
merged_df.loc[merged_df["ID"] == "094919364", "EP"] = "241"
merged_df.loc[merged_df["ID"] == "094966222", "SP"] = "16"
merged_df.loc[merged_df["ID"] == "094966222", "EP"] = "113"

In [15]:
# Unveraenderte Anmerkung sichern (fuer spaetere Rekonstruktion der Textbeziehungen)
merged_df["Anmerkung_original_backup"] = merged_df["Anmerkung"]


In [16]:
# Anmerkung: interne Verweise aufloesen (einzeln und mehrfach), unaufgeloeste als neue Zeilen anlegen
#
# Viele Anmerkungen (z.B. "Vgl. 1860. MB S. 1-3", "Auszug: 1837. MB S. 62-64;
# 95-103; 528-535", "Ebenda. S. 302-310") verweisen auf andere, bereits in
# merged_df enthaltene Zeilen. Ein Chunk kann dabei mehrere Seitenbereiche zu
# EINEM Jahr/Kuerzel nennen; jeder einzelne Bereich wird separat aufgeloest:
# existiert dafuer schon eine Zeile (eindeutig, gleicher A1, plausible Spanne
# -- MAX_SPAN_2 filtert Datenfehler wie ID 094931178 mit EP=2001 als Ziel
# aus), wird er aus der Anmerkung entfernt. Existiert noch keine, bleibt der
# Bereich in der Anmerkung stehen UND wird als neue Zeile angelegt (T1/A1 von
# der Quellzeile uebernommen, SR/BD/SP/EP aus dem Verweis). Chunks mit
# mehreren Jahres-/Kuerzel-Gruppen ohne " - "-Trennung (z.B. "...579-580.
# 1841. MB S. 101-102...") sind zu mehrdeutig und werden unveraendert gelassen.
#
# Neu angelegte Zeilen bekommen eine synthetische ID (Praefix "NEU-", keine
# echte GBV-ID) und keinen UR-Link, da der Zusammenhang zwischen gedruckter
# Seitenzahl und Scan-Bildnummer (pn=) nicht zuverlaessig rekonstruierbar ist.
# Herkunft_ID/Herkunft_Chunk verweisen auf die Quellzeile zur Kontrolle.

ABBR_LOOKUP_2 = {
    "MB": (["09-mon", "08-verh"], None),
    "SB I.": (["10-sitz"], "1"), "SB I": (["10-sitz"], "1"), "SB l.": (["10-sitz"], "1"),
    "SB II.": (["10-sitz"], "2"), "SB II": (["10-sitz"], "2"),
    "Hist.": (["02-hist"], None),
    "Phys. Abh.": (["07-abh"], None), "Math. Abh.": (["07-abh"], None),
    "Philol.-hist. Abh.": (["07-abh"], None), "Hist.-philol. Abh.": (["07-abh"], None),
    "Phys.-math. Abh.": (["07-abh"], None), "Philos.-hist. Abh.": (["07-abh"], None),
    "Hist.-philos. Abh.": (["07-abh"], None), "Philos. Abh.": (["07-abh"], None), "Abh.": (["07-abh"], None),
    "Mém. Classe de math.": (["05-mem"], None), "Mém. Classe de belles-lettres.": (["05-mem"], None),
    "Mém. Classe de philos. spécul.": (["05-mem"], None), "Mém. Classe de philos. expér.": (["05-mem"], None),
    "Mém.": (["05-mem"], None),
}
ABBR_ALTERNATION_2 = "|".join(re.escape(k) for k in sorted(ABBR_LOOKUP_2, key=len, reverse=True))
HEAD_RE = re.compile(
    rf"^(?P<label>Vgl\.?:?|Vergl\.?:?|vgl\.?:?|Auszug:?|Zusatz:?|Fortsetzung:?|Nachtrag:?)?\s*"
    rf"(?P<jahr>\d{{4}})\.\s*(?P<abbr>{ABBR_ALTERNATION_2})\s*S\.\s*(?P<tail>.+)$"
)
EMBEDDED_HEAD_RE = re.compile(rf"\d{{4}}\.\s*(?:{ABBR_ALTERNATION_2})\s*S\.")
EBENDA_HEAD_RE_2 = re.compile(r"^Ebenda\.?\s*S\.\s*(?P<tail>.+)$")
CHUNK_SPLIT_RE_2 = re.compile(r"\s-\s")
TOKEN_SPLIT_RE = re.compile(r"[;:]")
TOKEN_NUM_RE = re.compile(r"^(\d+)\s*-?\s*(\d+)?$")
MAX_SPAN_2 = 400

merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["EP_i"] = pd.to_numeric(merged_df["EP"], errors="coerce")


def surname_2(a1):
    return a1.split(",")[0].strip().lower() if pd.notna(a1) else ""


def find_target_2(sr_list, bd, p1, p2, own_id, own_author):
    sub = merged_df[merged_df["SR"].isin(sr_list) & (merged_df["BD"] == bd)]
    sub = sub[(sub["SP_i"] <= p2) & (sub["EP_i"] >= p1)]
    sub = sub[(sub["EP_i"] - sub["SP_i"]) <= MAX_SPAN_2]
    sub = sub[sub["ID"] != own_id]
    sub = sub[sub["A1"].apply(surname_2) == surname_2(own_author)]
    return sub


def resolve_sr_2(sr_list, bd):
    if len(sr_list) == 1:
        return sr_list[0]
    kandidaten = [sr for sr in sr_list if ((merged_df["SR"] == sr) & (merged_df["BD"] == bd)).any()]
    return kandidaten[0] if len(kandidaten) == 1 else None


neue_zeilen = []
anmerkung_teilweise = []
for _, row in merged_df.iterrows():
    text = row["Anmerkung"]
    if pd.isna(text):
        anmerkung_teilweise.append(np.nan)
        continue

    out_chunks = []
    for chunk in CHUNK_SPLIT_RE_2.split(text.strip()):
        s = chunk.strip()
        sr_list = bd = tail = head_text = None

        m = EBENDA_HEAD_RE_2.match(s)
        if m:
            if pd.notna(row["SR"]) and pd.notna(row["BD"]):
                sr_list, bd, tail = [row["SR"]], row["BD"], m.group("tail")
                head_text = "Ebenda. S."
        else:
            m2 = HEAD_RE.match(s)
            if m2 and not EMBEDDED_HEAD_RE.search(m2.group("tail")):
                jahr, abbr = m2.group("jahr"), m2.group("abbr")
                kandidaten_sr, teil = ABBR_LOOKUP_2[abbr]
                bd_try = f"{jahr}-{teil}" if teil else jahr
                sr = resolve_sr_2(kandidaten_sr, bd_try)
                if sr is not None:
                    sr_list, bd, tail = [sr], bd_try, m2.group("tail")
                    label = (m2.group("label") or "").strip()
                    head_text = (label + " " if label else "") + f"{jahr}. {abbr} S."

        if sr_list is None:
            out_chunks.append(chunk)
            continue

        kept_tokens, any_removed = [], False
        for tok in (t.strip() for t in TOKEN_SPLIT_RE.split(tail)):
            mnum = TOKEN_NUM_RE.match(tok.rstrip(" ."))
            if not mnum:
                kept_tokens.append(tok)
                continue
            p1 = int(mnum.group(1))
            p2 = int(mnum.group(2)) if mnum.group(2) else p1
            treffer = find_target_2(sr_list, bd, p1, p2, row["ID"], row["A1"])
            if len(treffer) == 1:
                any_removed = True
            else:
                kept_tokens.append(tok)
                neue_zeilen.append({
                    "ID": f"NEU-{row['ID']}-{len(neue_zeilen) + 1}",
                    "T1": row["T1"], "A1": row["A1"],
                    "SR": sr_list[0], "BD": bd, "SP": str(p1), "EP": str(p2),
                    "Herkunft_ID": row["ID"], "Herkunft_Chunk": tok,
                })

        if any_removed and kept_tokens:
            out_chunks.append(f"{head_text} " + "; ".join(kept_tokens))
        elif not any_removed:
            out_chunks.append(chunk)

    anmerkung_teilweise.append(" - ".join(out_chunks) if out_chunks else np.nan)

merged_df["Anmerkung"] = anmerkung_teilweise
merged_df = merged_df.drop(columns=["SP_i", "EP_i"])

if neue_zeilen:
    merged_df = pd.concat([merged_df, pd.DataFrame(neue_zeilen)], ignore_index=True)

print(f"Neu angelegte Zeilen aus unaufgelösten Teilverweisen: {len(neue_zeilen)}")


Neu angelegte Zeilen aus unaufgelösten Teilverweisen: 204


In [17]:
# Fortsetzung-Duplikate: SP/EP korrigieren
#
# Bei Artikeln mit Fortsetzung gibt es oft zwei (selten drei) Zeilen mit
# identischem T1/A1/Fortsetzung, aber unterschiedlichem UR (=unterschiedliche
# Scan-Bildnummer "pn="). Beide/alle tragen fälschlich dieselben SP/EP wie der
# Hauptteil. Die Zeile mit der niedrigsten pn ist der Hauptteil (SP/EP bleiben)
# -- alle anderen Zeilen der Gruppe bekommen die aus "Fortsetzung" geparsten
# Seitenangaben als SP/EP (überschreibt die falsch übernommenen Werte).

def extrahiere_pn(ur):
    m = re.search(r"pn=(\d+)", ur) if pd.notna(ur) else None
    return int(m.group(1)) if m else None


def parse_fortsetzung_seiten(text):
    t = re.sub(r"^ebenda\.\s*", "", text.strip(), flags=re.IGNORECASE).strip()
    werte = re.findall(r"\d+|[IVXLCDM]+", t)
    if not werte:
        return None, None
    if "-" in t and len(werte) >= 2:
        return werte[0], werte[1]
    if re.search(r"S\.?\s*$", t):
        # "<Wert> S." = Seitenzahl einer separat gezählten Fortsetzung (z.B. Anhang)
        return "1", werte[0]
    # blosse Zahl ohne "S." = Einzelseite, auf der die Fortsetzung steht
    return werte[0], werte[0]


merged_df["pn_tmp"] = merged_df["UR"].apply(extrahiere_pn)

korrigiert = 0
for _, gruppe in merged_df[merged_df["Fortsetzung"].notna()].groupby(["T1", "A1", "Fortsetzung"]):
    if len(gruppe) < 2:
        continue
    gruppe_sortiert = gruppe.sort_values("pn_tmp")
    fortsetzung_text = gruppe_sortiert.iloc[0]["Fortsetzung"]
    sp_neu, ep_neu = parse_fortsetzung_seiten(fortsetzung_text)
    if sp_neu is None:
        continue
    for idx in gruppe_sortiert.index[1:]:
        merged_df.loc[idx, "SP"] = sp_neu
        merged_df.loc[idx, "EP"] = ep_neu
        korrigiert += 1

merged_df = merged_df.drop(columns=["pn_tmp"])
print(f"SP/EP korrigiert bei {korrigiert} Fortsetzungs-Duplikaten")


SP/EP korrigiert bei 23 Fortsetzungs-Duplikaten


In [18]:
# Antwort: neue Zeilen fuer die Antworten auf Antrittsreden anlegen
#
# "Antwort" enthaelt Text wie "Antwort von BÖCKH: Ebenda. S. 217" -- eine
# Antwort auf die Antrittsrede der Quellzeile, im selben SR/BD (Ebenda), auf
# den genannten Seiten. Viele dieser Antworten sind aber schon als eigene
# Zeile katalogisiert (z.B. "Antwort auf die Antrittsreden von X, Y und Z"),
# oft weil ein/e Sekretär*in in einer Sitzung auf mehrere Antrittsreden
# gleichzeitig antwortete. Erst pruefen, ob eine solche Zeile schon existiert
# (SR/BD/Seiten-Overlap + Nachname), nur bei Nichtexistenz eine neue Zeile
# anlegen. Mehrere Quellzeilen mit identischem Antwort-Text (SR/BD/Name/
# Seiten) teilen sich EINE neue Zeile (wie die echten Vorbilder im Datensatz).
# Der volle Name der/des Antwortenden wird, wo moeglich, aus einer bereits
# bestehenden Zeile mit demselben Nachnamen uebernommen (sonst nur Nachname).
#
# Auf jeder Quellzeile verweist die neue Spalte "Antwort_ID" auf die
# (bestehende oder neu angelegte) Zielzeile.

ANTWORT_PAT = re.compile(
    r"^(Antwort von|Autwort von|Résponse de)\s+([^:.]+)[:.]\s*Ebenda\.?\s*S[.,]\s*(\d+)\s*-?\s*(\d+)?\s*$"
)
ANTWORT_ALIAS = {"ENCK": "ENCKE", "DUBOISREMOND": "DUBOISREYMOND"}


def antwort_norm_name(s):
    key = re.sub(r"[^A-ZÀ-Ü]", "", s.upper())
    return ANTWORT_ALIAS.get(key, key)


def antwort_surname_key(a1):
    return antwort_norm_name(a1.split(",")[0].strip()) if pd.notna(a1) else ""


merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["EP_i"] = pd.to_numeric(merged_df["EP"], errors="coerce")

antwort_zeilen = merged_df[merged_df["Antwort"].notna()]
name_map = {}
antwort_id = {}
unaufgeloeste_gruppen = {}

for idx, row in antwort_zeilen.iterrows():
    m = ANTWORT_PAT.match(row["Antwort"].strip())
    if not m:
        continue
    key_name = antwort_norm_name(m.group(2).strip())
    p1 = int(m.group(3))
    p2 = int(m.group(4)) if m.group(4) else p1
    sr, bd = row["SR"], row["BD"]

    sub = merged_df[(merged_df["SR"] == sr) & (merged_df["BD"] == bd)]
    sub = sub[(sub["SP_i"] <= p2) & (sub["EP_i"] >= p1)]
    sub = sub[sub["A1"].apply(antwort_surname_key) == key_name]

    if len(sub) == 1:
        antwort_id[idx] = sub.iloc[0]["ID"]
        name_map.setdefault(key_name, sub.iloc[0]["A1"])
    else:
        gruppen_key = (sr, bd, key_name, p1, p2)
        unaufgeloeste_gruppen.setdefault(gruppen_key, []).append(idx)

neue_zeilen = []
for n, ((sr, bd, key_name, p1, p2), idxs) in enumerate(unaufgeloeste_gruppen.items(), start=1):
    quellen = merged_df.loc[idxs]
    surnames = [a1.split(",")[0].strip() for a1 in quellen["A1"]]
    if len(surnames) == 1:
        t1 = f"Antwort auf die Antrittsrede von {surnames[0]}"
    else:
        t1 = f"Antwort auf die Antrittsreden von {', '.join(surnames[:-1])} und {surnames[-1]}"

    neu_id = f"NEU-ANTWORT-{n}"
    neue_zeilen.append({
        "ID": neu_id, "T1": t1, "A1": name_map.get(key_name, key_name.title()),
        "SR": sr, "BD": bd, "SP": str(p1), "EP": str(p2),
        "Herkunft_ID": ", ".join(quellen["ID"]),
        "Herkunft_Chunk": quellen.iloc[0]["Antwort"],
    })
    for idx in idxs:
        antwort_id[idx] = neu_id

merged_df["Antwort_ID"] = pd.Series(antwort_id)
merged_df = merged_df.drop(columns=["SP_i", "EP_i"])

if neue_zeilen:
    merged_df = pd.concat([merged_df, pd.DataFrame(neue_zeilen)], ignore_index=True)

print(f"Bereits bestehende Antwort-Zeilen verknuepft: {len(antwort_id) - len(neue_zeilen)}")
print(f"Neu angelegte Antwort-Zeilen: {len(neue_zeilen)}")


Bereits bestehende Antwort-Zeilen verknuepft: 60
Neu angelegte Antwort-Zeilen: 35


In [19]:
# Textbeziehungen: alle intertextuellen Beziehungen in einer Spalte buendeln
#
# Fasst alle bisher ueber SR/BD/Seiten aufgeloesten Beziehungen als ID-basierte
# "Typ: ID"-Paare zusammen (getrennt mit " | "), auf BEIDEN beteiligten Zeilen:
#   - Vgl/Auszug/Zusatz/Fortsetzung/Nachtrag/Verweis: aus Anmerkung_original_backup
#     neu ermittelt (die Zelle "Anmerkung: interne Verweise aufloesen" hat
#     diese IDs bereits berechnet, aber nicht persistiert -- daher hier eine
#     eigene, in sich abgeschlossene Neuberechnung anhand des Backups, ohne
#     die vorherige Zelle zu veraendern).
#   - Antwort/Antwort auf: aus der Spalte Antwort_ID.
#   - Fortsetzung/Fortsetzung von: aus den Fortsetzung-Duplikat-Gruppen (wie in
#     der Zelle "Fortsetzung-Duplikate", hier erneut anhand T1/A1/Fortsetzung/
#     UR ermittelt).
#
# Beziehungstyp ist aus Sicht der jeweiligen Zeile formuliert, z.B. bedeutet
# "Fortsetzung: <ID>" auf Zeile A "die Fortsetzung von A steht in <ID>", und
# "Fortsetzung von: <ID>" auf Zeile B "B ist die Fortsetzung von <ID>".

from collections import defaultdict

TB_ABBR_LOOKUP = {
    "MB": (["09-mon", "08-verh"], None),
    "SB I.": (["10-sitz"], "1"), "SB I": (["10-sitz"], "1"), "SB l.": (["10-sitz"], "1"),
    "SB II.": (["10-sitz"], "2"), "SB II": (["10-sitz"], "2"),
    "Hist.": (["02-hist"], None),
    "Phys. Abh.": (["07-abh"], None), "Math. Abh.": (["07-abh"], None),
    "Philol.-hist. Abh.": (["07-abh"], None), "Hist.-philol. Abh.": (["07-abh"], None),
    "Phys.-math. Abh.": (["07-abh"], None), "Philos.-hist. Abh.": (["07-abh"], None),
    "Hist.-philos. Abh.": (["07-abh"], None), "Philos. Abh.": (["07-abh"], None), "Abh.": (["07-abh"], None),
    "Mém. Classe de math.": (["05-mem"], None), "Mém. Classe de belles-lettres.": (["05-mem"], None),
    "Mém. Classe de philos. spécul.": (["05-mem"], None), "Mém. Classe de philos. expér.": (["05-mem"], None),
    "Mém.": (["05-mem"], None),
}
TB_ABBR_ALT = "|".join(re.escape(k) for k in sorted(TB_ABBR_LOOKUP, key=len, reverse=True))
TB_HEAD_RE = re.compile(
    rf"^(?P<label>Vgl\.?:?|Vergl\.?:?|vgl\.?:?|Auszug:?|Zusatz:?|Fortsetzung:?|Nachtrag:?)?\s*"
    rf"(?P<jahr>\d{{4}})\.\s*(?P<abbr>{TB_ABBR_ALT})\s*S\.\s*(?P<tail>.+)$"
)
TB_EMBEDDED_HEAD_RE = re.compile(rf"\d{{4}}\.\s*(?:{TB_ABBR_ALT})\s*S\.")
TB_EBENDA_HEAD_RE = re.compile(r"^Ebenda\.?\s*S\.\s*(?P<tail>.+)$")
TB_CHUNK_SPLIT_RE = re.compile(r"\s-\s")
TB_TOKEN_SPLIT_RE = re.compile(r"[;:]")
TB_TOKEN_NUM_RE = re.compile(r"^(\d+)\s*-?\s*(\d+)?$")
TB_MAX_SPAN = 400


def tb_surname(a1):
    return a1.split(",")[0].strip().lower() if pd.notna(a1) else ""


def tb_klassifiziere(label):
    # "Vgl"/"Vergl" wird bewusst NICHT als eigener Typ gefuehrt, sondern mit
    # dem allgemeinen "Verweis" zusammengefasst -- die Inhalte sind zu
    # uneinheitlich (mal Fortsetzungen ohne Teil1/Teil2-Kennzeichnung, mal
    # Duplikate, mal einfach dieselbe Stelle im Band), um einen praeziseren
    # eigenen Namen zu rechtfertigen.
    l = (label or "").lower().rstrip(": .").strip()
    if l.startswith("auszug"):
        return "Auszug"
    if l.startswith("zusatz"):
        return "Zusatz"
    if l.startswith("fortsetzung"):
        return "Fortsetzung"
    if l.startswith("nachtrag"):
        return "Nachtrag"
    return "Verweis"


merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["EP_i"] = pd.to_numeric(merged_df["EP"], errors="coerce")
tb_nicht_neu = merged_df[~merged_df["ID"].str.startswith("NEU-")]


def tb_find_target(sr_list, bd, p1, p2, own_id, own_author):
    sub = tb_nicht_neu[tb_nicht_neu["SR"].isin(sr_list) & (tb_nicht_neu["BD"] == bd)]
    sub = sub[(sub["SP_i"] <= p2) & (sub["EP_i"] >= p1)]
    sub = sub[(sub["EP_i"] - sub["SP_i"]) <= TB_MAX_SPAN]
    sub = sub[sub["ID"] != own_id]
    sub = sub[sub["A1"].apply(tb_surname) == tb_surname(own_author)]
    return sub


def tb_resolve_sr(sr_list, bd):
    if len(sr_list) == 1:
        return sr_list[0]
    kandidaten = [sr for sr in sr_list if ((tb_nicht_neu["SR"] == sr) & (tb_nicht_neu["BD"] == bd)).any()]
    return kandidaten[0] if len(kandidaten) == 1 else None


tb_neu_anmerkung = merged_df[merged_df["ID"].str.startswith("NEU-") & ~merged_df["ID"].str.startswith("NEU-ANTWORT")]
tb_herkunft_index = {
    (r["Herkunft_ID"], r["Herkunft_Chunk"]): r["ID"] for _, r in tb_neu_anmerkung.iterrows()
}

beziehungen_liste = []
for _, row in merged_df.iterrows():
    text = row["Anmerkung_original_backup"]
    if pd.isna(text):
        continue
    for chunk in TB_CHUNK_SPLIT_RE.split(text.strip()):
        s = chunk.strip()
        sr_list = bd = tail = label = None
        ebenda = False

        m = TB_EBENDA_HEAD_RE.match(s)
        if m:
            if pd.notna(row["SR"]) and pd.notna(row["BD"]):
                sr_list, bd, tail = [row["SR"]], row["BD"], m.group("tail")
                ebenda = True
        else:
            m2 = TB_HEAD_RE.match(s)
            if m2 and not TB_EMBEDDED_HEAD_RE.search(m2.group("tail")):
                jahr, abbr = m2.group("jahr"), m2.group("abbr")
                kand_sr, teil = TB_ABBR_LOOKUP[abbr]
                bd_try = f"{jahr}-{teil}" if teil else jahr
                sr = tb_resolve_sr(kand_sr, bd_try)
                if sr is not None:
                    sr_list, bd, tail = [sr], bd_try, m2.group("tail")
                    label = m2.group("label")

        if sr_list is None:
            continue
        typ = "Verweis" if ebenda else tb_klassifiziere(label)

        for tok in (t.strip() for t in TB_TOKEN_SPLIT_RE.split(tail)):
            mnum = TB_TOKEN_NUM_RE.match(tok.rstrip(" ."))
            if not mnum:
                continue
            p1 = int(mnum.group(1))
            p2 = int(mnum.group(2)) if mnum.group(2) else p1
            treffer = tb_find_target(sr_list, bd, p1, p2, row["ID"], row["A1"])
            if len(treffer) == 1:
                beziehungen_liste.append((row["ID"], typ, treffer.iloc[0]["ID"]))
            else:
                ziel = tb_herkunft_index.get((row["ID"], tok))
                if ziel:
                    beziehungen_liste.append((row["ID"], typ, ziel))

# Antwort_ID -> Beziehung
for _, r in merged_df[merged_df["Antwort_ID"].notna()].iterrows():
    beziehungen_liste.append((r["ID"], "Antwort", r["Antwort_ID"]))


# Fortsetzung-Duplikate -> Beziehung
def tb_extrahiere_pn(ur):
    m = re.search(r"pn=(\d+)", ur) if pd.notna(ur) else None
    return int(m.group(1)) if m else None


merged_df["pn_tmp"] = merged_df["UR"].apply(tb_extrahiere_pn)
for _, g in merged_df[merged_df["Fortsetzung"].notna()].groupby(["T1", "A1", "Fortsetzung"]):
    if len(g) < 2:
        continue
    g_sortiert = g.sort_values("pn_tmp")
    haupt_id = g_sortiert.iloc[0]["ID"]
    for _, r in g_sortiert.iloc[1:].iterrows():
        beziehungen_liste.append((haupt_id, "Fortsetzung", r["ID"]))

# Übersetzung/Original -> Beziehung
#
# "Übersetzung unter dem Titel: X. Jahr. Kuerzel S. Seiten" (bzw. "Deutsch
# unter dem Titel"/"In franzoesischer Uebersetzung unter dem Titel") auf
# Zeile A bedeutet: eine Uebersetzung von A steht unter dem Titel X an
# dieser Stelle -> Beziehung (A, "Uebersetzung", Ziel). "Original unter dem
# Titel" (bzw. "Uebersetzung von X") bedeutet umgekehrt: A IST die
# Uebersetzung, das Original steht an dieser Stelle -> Quelle/Ziel vertauscht,
# damit "Uebersetzung"/"Uebersetzung von" in beiden Faellen dieselbe
# Bedeutung behalten.

TB_UEB_CITATION_RE = re.compile(rf"(\d{{4}})(?:/\d{{2,4}})?\.\s*({TB_ABBR_ALT})\s*S\.\s*(\d+)(?:\s*-\s*(\d+))?")
TB_UEB_SWAP_LABELS = ("Original unter dem Titel", "Übersetzung von")

for _, row in merged_df[merged_df["Übersetzung/Original"].notna()].iterrows():
    text = row["Übersetzung/Original"]
    swap = text.startswith(TB_UEB_SWAP_LABELS)
    for m in TB_UEB_CITATION_RE.finditer(text):
        jahr, abbr, p1s, p2s = m.groups()
        p1 = int(p1s)
        p2 = int(p2s) if p2s else p1
        kand_sr, teil = TB_ABBR_LOOKUP[abbr]
        bd = f"{jahr}-{teil}" if teil else jahr
        sr = tb_resolve_sr(kand_sr, bd)
        if sr is None:
            continue
        treffer = tb_find_target([sr], bd, p1, p2, row["ID"], row["A1"])
        if len(treffer) == 1:
            ziel = treffer.iloc[0]["ID"]
            paar = (ziel, row["ID"]) if swap else (row["ID"], ziel)
            beziehungen_liste.append((paar[0], "Übersetzung", paar[1]))

# Die Rueckrichtung muss ebenfalls benennen, WAS der jeweilige Zieltitel ist
# (nicht nur "X von" als Verweis auf die Vorwaertsrichtung):
#   - Uebersetzung -> Original (Zielzeile IST das Original)
#   - Auszug -> Volltext (Zielzeile IST die vollstaendige Fassung)
#   - Fortsetzung/Nachtrag -> Hauptteil (Zielzeile IST der Hauptteil, zu dem
#     die aktuelle Zeile Fortsetzung bzw. nachtraeglicher Zusatz ist)
#   - Antwort -> Antrittsrede (Zielzeile IST die beantwortete Antrittsrede)
#   - Verweis bleibt in beide Richtungen "Verweis" (die Beziehung ist zu
#     uneinheitlich fuer eine gerichtete Bezeichnung)
TB_REVERSE = {
    "Auszug": "Volltext", "Zusatz": "Zusatz von",
    "Fortsetzung": "Hauptteil", "Nachtrag": "Hauptteil",
    "Verweis": "Verweis", "Antwort": "Antrittsrede", "Übersetzung": "Original",
}

tb_dict = defaultdict(list)
for a, typ, b in dict.fromkeys(beziehungen_liste):
    tb_dict[a].append(f"{typ}: {b}")
    tb_dict[b].append(f"{TB_REVERSE[typ]}: {a}")

textbeziehungen = {k: " | ".join(dict.fromkeys(v)) for k, v in tb_dict.items()}
merged_df["Textbeziehungen"] = merged_df["ID"].map(textbeziehungen)
merged_df = merged_df.drop(columns=["SP_i", "EP_i", "pn_tmp", "Anmerkung_original_backup"])

print(f"Zeilen mit Textbeziehungen: {merged_df['Textbeziehungen'].notna().sum()}")


Zeilen mit Textbeziehungen: 984


In [20]:
# UR fuer neu angelegte Zeilen (NEU-*) aus der Seitenzahl herleiten
#
# Der Link enthaelt "pn=<Bildnummer>" -- die Nummer des Scan-Bildes, die von
# der gedruckten Seitenzahl (SP) um einen Versatz abweicht (Titelblaetter,
# Inhaltsverzeichnisse etc.). Dieser Versatz ist nicht im ganzen Band
# konstant (mehrere Test-Ansaetze mit einem einzigen Band-weiten Versatz
# bzw. mit Interpolation zwischen den Nachbarn schnitten schlechter ab), aber
# lokal meist stabil. Deshalb: fuer jede NEU-*-Zeile den existierenden
# Artikel im selben SR/BD mit der zahlenmaessig naechstgelegenen SP suchen
# und dessen Versatz (pn - SP) uebernehmen. Das ist eine Schaetzung, kein
# garantiert korrekter Link -- daher zusaetzlich als "UR_geschaetzt"
# markiert, damit das im Zweifel nachvollzogen/geprueft werden kann.

merged_df["SP_i"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df["pn_i"] = merged_df["UR"].str.extract(r"pn=(\d+)")[0]

bekannt = merged_df[
    ~merged_df["ID"].str.startswith("NEU-") & merged_df["UR"].notna() & merged_df["pn_i"].notna()
].copy()
bekannt["pn_i"] = bekannt["pn_i"].astype(int)

ur_geschaetzt = {}
for idx, row in merged_df[merged_df["ID"].str.startswith("NEU-")].iterrows():
    if pd.isna(row["SR"]) or pd.isna(row["BD"]) or pd.isna(row["SP_i"]):
        continue
    kandidaten = bekannt[(bekannt["SR"] == row["SR"]) & (bekannt["BD"] == row["BD"])]
    if kandidaten.empty:
        continue
    nachbar = kandidaten.loc[(kandidaten["SP_i"] - row["SP_i"]).abs().idxmin()]

    versatz = nachbar["pn_i"] - nachbar["SP_i"]
    pn_neu = max(1, round(row["SP_i"] + versatz))
    pn_string_alt = re.search(r"pn=(\d+)", nachbar["UR"]).group(1)
    pn_string_neu = str(int(pn_neu)).zfill(len(pn_string_alt))

    merged_df.loc[idx, "UR"] = re.sub(r"pn=\d+", f"pn={pn_string_neu}", nachbar["UR"])
    ur_geschaetzt[row["ID"]] = "ja"

merged_df["UR_geschaetzt"] = merged_df["ID"].map(ur_geschaetzt)
merged_df = merged_df.drop(columns=["SP_i", "pn_i"])

print(f"UR geschaetzt fuer {len(ur_geschaetzt)} von "
      f"{merged_df['ID'].str.startswith('NEU-').sum()} neu angelegten Zeilen")


UR geschaetzt fuer 221 von 239 neu angelegten Zeilen


In [21]:
merged_df = merged_df.drop(columns=["Anmerkung", "Fortsetzung", "Antwort", "Herkunft_ID", "Herkunft_Chunk", "Antwort_ID", "UR_geschaetzt", "Übersetzung/Original"])

In [22]:
# Titel normalisieren: Anfuehrungszeichen am Anfang/Ende entfernen, ersten
# Buchstaben grossschreiben.
#
# Wirkt sich damit auch auf die Textbeziehungen-Anzeige aus, da diese im
# Enrich-Schritt (03_manifest_enrich.ipynb, beziehungen_html()) den Titel der
# jeweiligen Zielzeile hinter dem Beziehungstyp einsetzt -- ein normalisierter
# Titel liefert dort automatisch ein grossgeschriebenes erstes Wort ohne
# unpassende Anfuehrungszeichen.

QUOTE_CHARS = "\"'\u201e\u201c\u2018\u2019\u201d`\u00b4"

merged_df["T1"] = merged_df["T1"].str.strip(QUOTE_CHARS).str.strip()
merged_df["T1"] = merged_df["T1"].apply(
    lambda s: s[0].upper() + s[1:] if isinstance(s, str) and s else s
)


In [23]:
# AK-Geschichte kuratieren
merged_df = merged_df[~merged_df['UR'].str.contains('digitalequel', na=False)]

ak_gesch_mask = merged_df['SR'] == 'ak-gesch'

# Seitenzahlen und Band sind bei ak-gesch nicht sinnvoll (kein Sammelband mit
# durchgezaehlten Abhandlungen wie bei den anderen Schriftenreihen)
merged_df.loc[ak_gesch_mask, ['SP', 'EP']] = None

# Titel/Autor fuer die beiden unbetitelten Bartholmess-Baende nachtragen
merged_df.loc[ak_gesch_mask & (merged_df['T1'] == 'Teil 1'), ['T1', 'A1']] = [
    "Histoire philosophique de l'Académie de Prusse depuis Leibniz jusqu'a Schelling, particul. sous Frédéric-le-Grand [Teil 1]",
    "Bartholmèss, Christian",
]
merged_df.loc[ak_gesch_mask & (merged_df['T1'] == 'Teil 2'), ['T1', 'A1']] = [
    "Histoire philosophique de l'Académie de Prusse depuis Leibniz jusqu'a Schelling, particul. sous Frédéric-le-Grand [Teil 2]",
    "Bartholmèss, Christian",
]

# Erscheinungsjahr fuer den Bildband nachtragen
merged_df.loc[ak_gesch_mask & merged_df['T1'].str.contains('Bildnisse berühmter', na=False), 'PY'] = "1950"

# Eckige Klammern aus dem Erscheinungsjahr entfernen (z.B. "[1900]" -> "1900")
merged_df.loc[ak_gesch_mask, 'PY'] = merged_df.loc[ak_gesch_mask, 'PY'].str.replace(r'[\[\]]', '', regex=True)

In [24]:
merged_df["SP_sort"] = pd.to_numeric(merged_df["SP"], errors="coerce")
merged_df = merged_df.sort_values(by=["SR", "BD", "SP_sort"])
merged_df = merged_df.drop(columns=["SP_sort"])

# Sprechende Spaltennamen fuer den Export (nur hier, damit die Zellen
# oben (SP/EP/T1/A1/SR/BD/UR) unveraendert weiterfunktionieren)
merged_df = merged_df.rename(columns={
    "T1": "Titel",
    "A1": "Autor",
    "UR": "Link",
    "SR": "Schriftenreihe",
    "BD": "Band",
    "SP": "Startseite",
    "EP": "Endseite",
    "PY": "Jahr",
})

merged_df.to_csv("../../data/metadata/curated/merged_df.csv", index=False, encoding="utf-8-sig")